In [1]:
pip install openpyxl

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import re
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from datasets import Dataset


In [3]:
df = pd.read_excel("Reddit_Combined_Data.xlsx")

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    return text.strip()

def build_text(row):
    title = clean_text(row.get("title", ""))
    body = clean_text(row.get("selftext", ""))
    return f"{title} {body}".strip()

df["text"] = df.apply(build_text, axis=1)

# Remove weak samples (paper-style)
df = df[df["text"].str.split().str.len() >= 8]
print("Dataset after cleaning:", df.shape)


Dataset after cleaning: (799, 13)


In [4]:
def map_severity(raw):
    if pd.isna(raw):
        return "Not mentioned"
    s = str(raw).lower()

    if any(k in s for k in ["panic", "anxiety", "anxious"]):
        return "Anxiety/Panic"
    if any(k in s for k in ["depress", "hopeless", "worthless"]):
        return "Depression"
    if any(k in s for k in ["ptsd", "trauma", "suicidal", "self harm"]):
        return "Severe/PTSD"
    if any(k in s for k in ["lonely", "isolated"]):
        return "Loneliness"
    if any(k in s for k in ["addict", "alcohol", "drug"]):
        return "Substance"
    return "Not mentioned"

df["severity_merged"] = df["Perceived severity"].apply(map_severity)
print(df["severity_merged"].value_counts())


severity_merged
Depression       343
Anxiety/Panic    277
Not mentioned    137
Substance         23
Severe/PTSD       19
Name: count, dtype: int64


In [5]:
def map_self_eff(raw):
    s = str(raw).lower()
    if s in ["low", "very low"]:
        return "Low"
    if s in ["high", "very high"]:
        return "High"
    return "Low"

df["self_eff_binary"] = df["Self-efficacy"].apply(map_self_eff)


In [6]:
def map_cue(raw):
    s = str(raw).lower()
    if "help" in s or "treatment" in s:
        return "Action"
    if "information" in s:
        return "Information"
    return "None"

df["cue_group"] = df["Cue to Action"].apply(map_cue)


In [7]:
def map_root(raw):
    s = str(raw).lower()
    if "trauma" in s:
        return "Trauma"
    if "stress" in s:
        return "Stress"
    if "substance" in s or "drug" in s:
        return "Substance"
    if "family" in s:
        return "Family"
    return "Other"

df["root_cause_final"] = df["Root cause"].apply(map_root)


In [8]:
def map_benefit(raw):
    s = str(raw).lower()
    if "support" in s:
        return "Support"
    if "treatment" in s:
        return "Treatment"
    return "None"

df["benefit_final"] = df["Perceived benefits"].apply(map_benefit)


In [9]:
TASKS = {
    "severity": "severity_merged",
    "self_efficacy": "self_eff_binary",
    "cue_to_action": "cue_group",
    "root_cause": "root_cause_final",
    "perceived_benefits": "benefit_final"
}


In [10]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )


In [15]:
results = []

for task, col in TASKS.items():
    print(f"\n=== Training {task} ===")

    # ----------------------------
    # Filter valid samples
    # ----------------------------
    df_task = df.dropna(subset=[col]).copy()

    if df_task[col].value_counts().min() < 5:
        print("Skipping — insufficient samples")
        continue

    # ----------------------------
    # Encode labels
    # ----------------------------
    le = LabelEncoder()
    df_task["labels"] = le.fit_transform(df_task[col])   # 🔑 labels (NOT label)

    # ----------------------------
    # Train / Test split
    # ----------------------------
    X_train, X_test, y_train, y_test = train_test_split(
        df_task["text"].tolist(),
        df_task["labels"].tolist(),
        test_size=0.2,
        stratify=df_task["labels"],
        random_state=42
    )

    # ----------------------------
    # HuggingFace Dataset
    # ----------------------------
    train_ds = Dataset.from_pandas(
        pd.DataFrame({"text": X_train, "labels": y_train})
    )
    test_ds = Dataset.from_pandas(
        pd.DataFrame({"text": X_test, "labels": y_test})
    )

    train_ds = train_ds.map(tokenize, batched=True)
    test_ds = test_ds.map(tokenize, batched=True)

    train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    test_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

    # ----------------------------
    # Model
    # ----------------------------
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(le.classes_)
    )

    model.config.problem_type = "single_label_classification"

    # ----------------------------
    # Training Arguments
    # ----------------------------
    training_args = TrainingArguments(
        output_dir=f"./results/{task}",
        per_device_train_batch_size=8,
        num_train_epochs=5,
        learning_rate=2e-5,
        logging_steps=50,
        report_to="none",
        do_train=True,
        do_eval=False,                 # 🔑 avoid eval_strategy issues
        remove_unused_columns=False
    )

    # ----------------------------
    # Trainer
    # ----------------------------
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds
    )

    # ----------------------------
    # Train
    # ----------------------------
    trainer.train()

    # ----------------------------
    # Manual Evaluation
    # ----------------------------
    preds = trainer.predict(test_ds)
    y_pred = preds.predictions.argmax(axis=1)

    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro")
    weighted_f1 = f1_score(y_test, y_pred, average="weighted")

    print(f"Accuracy     : {acc:.4f}")
    print(f"Macro F1     : {macro_f1:.4f}")
    print(f"Weighted F1  : {weighted_f1:.4f}")

    results.append([task, acc, macro_f1, weighted_f1])



=== Training severity ===


Map:   0%|          | 0/639 [00:00<?, ? examples/s]

Map:   0%|          | 0/160 [00:00<?, ? examples/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
50,1.288300
100,1.184100
150,1.021800
200,0.887600
250,0.879200
300,0.814500
350,0.734600
400,0.668700


...

Map:   0%|          | 0/639 [00:00<?, ? examples/s]

Map:   0%|          | 0/160 [00:00<?, ? examples/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
50,0.892900
100,0.858000
150,0.743200
200,0.689500
250,0.626000
300,0.556000
350,0.519200
400,0.466300


Accuracy     : 0.6813
Macro F1     : 0.5142
Weighted F1  : 0.6645


In [16]:
results_df = pd.DataFrame(
    results,
    columns=["Task", "Accuracy", "Macro F1", "Weighted F1"]
)

results_df


,Task,Accuracy,Macro F1,Weighted F1
0,severity,0.59375,0.316134,0.552154
1,self_efficacy,1.00000,1.000000,1.000000
2,cue_to_action,0.78125,0.519033,0.765178
3,root_cause,0.77500,0.755952,0.769345
4,perceived_benefits,0.68125,0.514178,0.664512
